In [6]:
import os
import sys
from pathlib import Path

REPO_PATH = Path("/content/Generative-Models-Comparation")
if REPO_PATH.exists():
    os.chdir(REPO_PATH)

ROOT_DIR = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import torch
import pandas as pd
from src.config import Config
from src.utils.seed import set_seed
from src.utils.io_helpers import load_checkpoint, load_json, save_metrics_table
from src.dataset import get_dataloaders
from src.models.vanilla_ae import VanillaAutoencoder
from src.models.sparse_ae import SparseAutoencoder
from src.models.vae import VariationalAutoencoder
from src.models.vanilla_gan import VanillaGenerator
from src.models.dcgan import DCGANGenerator
from src.models.ddpm import SmallUNet, DDPM
from src.evaluation.metrics import compute_fid_score

set_seed(Config.SEED)
Config.ensure_dirs()
_, _, test_loader = get_dataloaders(
    batch_size=Config.BATCH_SIZE,
    seed=Config.SEED,
    num_workers=2 if torch.cuda.is_available() else 0,
)

# Muat seluruh bobot model terlatih (01 - 07)
vanilla_ae = VanillaAutoencoder(Config.INPUT_DIM, 256, Config.AE_LATENT_DIM).to(Config.DEVICE)
load_checkpoint(vanilla_ae, Config.CHECKPOINT_DIR / "vanilla_ae.pth", Config.DEVICE)
vanilla_ae.eval()

sparse_ae = SparseAutoencoder(Config.INPUT_DIM, Config.SPARSE_AE_HIDDEN_DIM, Config.SPARSE_AE_LATENT_DIM).to(Config.DEVICE)
load_checkpoint(sparse_ae, Config.CHECKPOINT_DIR / "sparse_ae.pth", Config.DEVICE)
sparse_ae.eval()

vae = VariationalAutoencoder(Config.INPUT_DIM, 256, Config.VAE_LATENT_DIM).to(Config.DEVICE)
load_checkpoint(vae, Config.CHECKPOINT_DIR / "vae.pth", Config.DEVICE)
vae.eval()

vanilla_gan_gen = VanillaGenerator(Config.VANILLA_GAN_NOISE_DIM, Config.INPUT_DIM).to(Config.DEVICE)
load_checkpoint(vanilla_gan_gen, Config.CHECKPOINT_DIR / "vanilla_gan_gen.pth", Config.DEVICE)
vanilla_gan_gen.eval()

dcgan_gen = DCGANGenerator(Config.DCGAN_NOISE_DIM, Config.DCGAN_FEATURE_MAPS, Config.IMG_CHANNELS).to(Config.DEVICE)
load_checkpoint(dcgan_gen, Config.CHECKPOINT_DIR / "dcgan_gen.pth", Config.DEVICE)
dcgan_gen.eval()

ddpm_unet = SmallUNet(Config.IMG_CHANNELS, Config.DDPM_BASE_CHANNELS, 128)
ddpm = DDPM(ddpm_unet, Config.DDPM_TIMESTEPS, Config.DDPM_BETA_START, Config.DDPM_BETA_END).to(Config.DEVICE)
load_checkpoint(ddpm, Config.CHECKPOINT_DIR / "ddpm.pth", Config.DEVICE)
ddpm.eval()

# Hitung FID Score pada Test Set (1.024 sampel sintetis)
NUM_FID_SAMPLES = 1024
print("Computing FID Scores on Test Set...")
fid_vae = compute_fid_score(test_loader, lambda bs: vae.sample(bs, Config.DEVICE), Config.DEVICE, NUM_FID_SAMPLES)
print(f"  VAE FID         : {fid_vae}")

fid_vgan = compute_fid_score(test_loader, lambda bs: vanilla_gan_gen.sample(bs, Config.DEVICE), Config.DEVICE, NUM_FID_SAMPLES)
print(f"  Vanilla GAN FID : {fid_vgan}")

fid_dcgan = compute_fid_score(test_loader, lambda bs: dcgan_gen.sample(bs, Config.DEVICE), Config.DEVICE, NUM_FID_SAMPLES)
print(f"  DCGAN FID       : {fid_dcgan}")

fid_ddpm = compute_fid_score(test_loader, lambda bs: ddpm.sample(bs, Config.DEVICE), Config.DEVICE, NUM_FID_SAMPLES, batch_size=256)
print(f"  DDPM FID        : {fid_ddpm}")

Computing FID Scores on Test Set...
  VAE FID         : 95.9605
  Vanilla GAN FID : 142.0776
  DCGAN FID       : 29.6447
  DDPM FID        : 34.0873


In [8]:
res_vae_ae = load_json(Config.METRICS_DIR / "vanilla_ae_results.json")
res_sparse_ae = load_json(Config.METRICS_DIR / "sparse_ae_results.json")
res_vae = load_json(Config.METRICS_DIR / "vae_results.json")
res_vgan = load_json(Config.METRICS_DIR / "vanilla_gan_results.json")
res_dcgan = load_json(Config.METRICS_DIR / "dcgan_results.json")
res_ddpm = load_json(Config.METRICS_DIR / "ddpm_results.json")
res_mlp = load_json(Config.METRICS_DIR / "mlp_limitations_metrics.json")

records = [
    {
        "Arsitektur Model": "Vanilla Autoencoder CPU",
        "Dim Laten": res_vae_ae["latent_dim"],
        "Training Time / Epoch (s)": res_vae_ae["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_vae_ae["inference_latency_ms_batch64"],
        "Test MSE": res_vae_ae["test_metrics"]["mse"],
        "Test PSNR (dB)": res_vae_ae["test_metrics"]["psnr"],
        "Test SSIM": res_vae_ae["test_metrics"]["ssim"],
        "FID Score": "N/A (Latent Voids)",
        "Kestabilan Training": "Stabil / Konvergen",
    },
    {
        "Arsitektur Model": "Sparse Autoencoder CPU",
        "Dim Laten": f"{res_sparse_ae['latent_dim']} (L1={res_sparse_ae['l1_lambda']})",
        "Training Time / Epoch (s)": res_sparse_ae["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_sparse_ae["inference_latency_ms_batch64"],
        "Test MSE": res_sparse_ae["test_metrics"]["mse"],
        "Test PSNR (dB)": res_sparse_ae["test_metrics"]["psnr"],
        "Test SSIM": res_sparse_ae["test_metrics"]["ssim"],
        "FID Score": "N/A (Latent Voids)",
        "Kestabilan Training": "Stabil / Konvergen",
    },
    {
        "Arsitektur Model": "Variational Autoencoder (VAE) CPU",
        "Dim Laten": res_vae["latent_dim"],
        "Training Time / Epoch (s)": res_vae["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_vae["inference_latency_ms_batch64"],
        "Test MSE": res_vae["test_metrics"]["mse"],
        "Test PSNR (dB)": res_vae["test_metrics"]["psnr"],
        "Test SSIM": res_vae["test_metrics"]["ssim"],
        "FID Score": fid_vae,
        "Kestabilan Training": "Stabil / Konvergen",
    },
    {
        "Arsitektur Model": "Vanilla GAN (MLP) CPU",
        "Dim Laten": res_vgan["latent_dim"],
        "Training Time / Epoch (s)": res_vgan["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_vgan["inference_latency_ms_batch64"],
        "Test MSE": "N/A (Tanpa Encoder)",
        "Test PSNR (dB)": "N/A",
        "Test SSIM": "N/A",
        "FID Score": fid_vgan,
        "Kestabilan Training": "Rentan Osilasi / Mode Collapse",
    },
    {
        "Arsitektur Model": "Deep Convolutional GAN (DCGAN) CPU",
        "Dim Laten": res_dcgan["latent_dim"],
        "Training Time / Epoch (s)": res_dcgan["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_dcgan["inference_latency_ms_batch64"],
        "Test MSE": "N/A (Tanpa Encoder)",
        "Test PSNR (dB)": "N/A",
        "Test SSIM": "N/A",
        "FID Score": fid_dcgan,
        "Kestabilan Training": "Cukup Stabil (Sensitif Arsitektur)",
    },
    {
        "Arsitektur Model": "DDPM (U-Net, T=1000) T4 GPU",
        "Dim Laten": "28x28 (Dimensi Citra)",
        "Training Time / Epoch (s)": res_ddpm["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_ddpm["inference_latency_ms_batch64"],
        "Test MSE": "N/A (Denoising Model)",
        "Test PSNR (dB)": "N/A",
        "Test SSIM": "N/A",
        "FID Score": fid_ddpm,
        "Kestabilan Training": "Sangat Stabil",
    },
]

df_table = save_metrics_table(records, Config.METRICS_DIR / "final_comparative_table.csv")
display(df_table)

print("\n=== REFERENSI MLP FROZEN DECODER (BAGIAN III LAPORAN) ===")
print(f"MLP Classifier Test Accuracy: {res_mlp['classifier_test_acc']}%")
print(f"Frozen MLP Decoder Metrics  : {res_mlp['frozen_decoder_metrics']}")

,Arsitektur Model,Dim Laten,Training Time / Epoch (s),Latency (ms / batch 64),Test MSE,Test PSNR (dB),Test SSIM,FID Score,Kestabilan Training
0,Vanilla Autoencoder CPU,64,8.1267,0.7173,0.008242,20.8399,0.836,N/A (Latent Voids),Stabil / Konvergen
1,Sparse Autoencoder CPU,64 (L1=0.0001),11.8187,0.5817,0.008236,20.8431,0.8356,N/A (Latent Voids),Stabil / Konvergen
2,Variational Autoencoder (VAE) CPU,64,13.2806,0.8303,0.015321,18.147,0.7416,95.9605,Stabil / Konvergen
3,Vanilla GAN (MLP) CPU,100,24.0782,4.3593,N/A (Tanpa Encoder),N/A,N/A,142.0776,Rentan Osilasi / Mode Collapse
4,Deep Convolutional GAN (DCGAN) CPU,100,195.6500,41.1728,N/A (Tanpa Encoder),N/A,N/A,29.6447,Cukup Stabil (Sensitif Arsitektur)
5,"DDPM (U-Net, T=1000) T4 GPU",28x28 (Dimensi Citra),44.7284,16435.8387,N/A (Denoising Model),N/A,N/A,34.0873,Sangat Stabil



=== REFERENSI MLP FROZEN DECODER (BAGIAN III LAPORAN) ===
MLP Classifier Test Accuracy: 88.8857%
Frozen MLP Decoder Metrics  : {'mse': 0.017125, 'psnr': 17.6636, 'ssim': 0.7124}


In [9]:
from src.evaluation.visualizations import (
    generate_figure_1_reconstructions,
    generate_figure_2_random_sampling,
    generate_figure_3_mlp_limitations,
    generate_figure_4_latent_interpolation,
    generate_figure_5_loss_and_ddpm_trajectory,
)

mlp_visuals = torch.load(Config.METRICS_DIR / "mlp_limitations_visuals.pt", map_location="cpu", weights_only=False)
ddpm_snapshots = torch.load(Config.METRICS_DIR / "ddpm_trajectory_snapshots.pt", map_location="cpu", weights_only=False)

histories = {
    "vanilla_ae": res_vae_ae["history"],
    "sparse_ae": res_sparse_ae["history"],
    "vae": res_vae["history"],
    "vanilla_gan": res_vgan["history"],
    "dcgan": res_dcgan["history"],
    "ddpm": res_ddpm["history"],
}

generate_figure_1_reconstructions(
    vanilla_ae, sparse_ae, vae, test_loader, Config.CLASS_NAMES, Config.DEVICE,
    Config.FIGURES_DIR / "fig1_reconstruction_grid.png",
)
generate_figure_2_random_sampling(
    vae, vanilla_gan_gen, dcgan_gen, ddpm, Config.DEVICE,
    Config.FIGURES_DIR / "fig2_random_sampling_grid.png",
)
generate_figure_3_mlp_limitations(
    mlp_visuals, vanilla_ae, Config.CLASS_NAMES, Config.DEVICE,
    Config.FIGURES_DIR / "fig3_mlp_limitations.png",
)
generate_figure_4_latent_interpolation(
    vanilla_ae, sparse_ae, vae, vanilla_gan_gen, dcgan_gen, test_loader, Config.DEVICE,
    Config.FIGURES_DIR / "fig4_latent_interpolation.png",
)
generate_figure_5_loss_and_ddpm_trajectory(
    histories, ddpm_snapshots,
    Config.FIGURES_DIR / "fig5_training_curves_and_ddpm_trajectory.png",
)

print("Selesai! Kelima figur disimpan di:", Config.FIGURES_DIR)

Selesai! Kelima figur disimpan di: /content/Generative-Models-Comparation/results/figures
